# EMPOWER KMOS — Spectroscopic-Redshift Audit for Target Selection
**Field**: COSMOS, GOODS-S **|** **Programme**: ESO 26107 (LP) **|** **Period**: P117+

## Purpose
This notebook quantifies, per field and per EMPOWER redshift tier, **(i)** what
spectroscopic redshift information is currently in hand and **(ii)** where
information is still missing or only available as a photometric estimate.
The audit is intended as the data foundation for EMPOWER target selection
($M_\star \geq 10^{10}\,M_\odot$) in the three tiers $z \simeq 0.75, 1.6, 2.3$ and
for the KARMA simultaneity constraint that puts H$\beta$+H$\alpha$ inside the IZ band
for $0.60 < z < 0.6325$.

## Method
For each field we adopt one **baseline catalogue** (the most complete public
compilation) and then quantify what is added by each of several recently
released or unmerged surveys via a 1 arcsec sky cross-match. Outputs are
candidate counts per tier, redshift-distribution figures, and per-survey
agreement diagnostics.

| Field   | Baseline                              | Add-on surveys cross-checked            |
|---------|---------------------------------------|-----------------------------------------|
| COSMOS  | Khostovan+25 DR1.1 unique (262 k z)  | DJA NIRSpec v4.4, DEVILS DR1 D10        |
| GOODS-S | ASTRODEEP-GS43 (Merlin+21, 35 k)      | DJA NIRSpec, MUSE-Wide DR1, MUSE-UDF DR2, VANDELS DR4 |

## Caveat (read before using any cross-matched count below)
Cross-matches in this notebook are **purely positional**, with a 1 arcsec
radius. They are sufficient to *count* the size of the gap each new survey
fills but **not** sufficient for science-quality identification of individual
objects: MSA/IFU programmes target compact faint sources that frequently
neighbour brighter foreground galaxies. Pairs flagged here for cross-survey
disagreement should be vetted manually (image-matched IDs, line identifications)
in a follow-up notebook before being relied on for KMOS selection.

## Outline
1. **Setup** — constants, helpers, catalogue paths.
2. **COSMOS** — Khostovan baseline + DJA & DEVILS gap analyses.
3. **GOODS-S** — ASTRODEEP baseline + a unified compilation merging DJA, MUSE-Wide,
   MUSE-UDF, VANDELS DR4.
4. **Synthesis** — combined per-tier inventory and prioritised next-survey list.


## 1 · Setup

Imports, plotting style, and the survey-wide constants used everywhere below
(EMPOWER tier windows, KMOS bands, line wavelengths, OH-forest envelopes,
file paths).


In [ ]:
"""Standard imports and plotting style."""
import os
import warnings
from dataclasses import dataclass
from typing import Sequence

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

from astropy.table import Table, vstack
from astropy.coordinates import SkyCoord
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

warnings.filterwarnings("ignore", category=UserWarning)
np.set_printoptions(suppress=True, linewidth=140)

# Q1-journal style — tight, readable, colour-blind safe where applicable.
plt.rcParams.update({
    "figure.dpi":       110,
    "savefig.dpi":      150,
    "savefig.bbox":     "tight",
    "font.size":        10,
    "axes.titlesize":   11,
    "axes.labelsize":   10,
    "legend.fontsize":  9,
    "xtick.labelsize":  9,
    "ytick.labelsize":  9,
    "axes.grid":        True,
    "grid.alpha":       0.25,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "lines.linewidth":  1.4,
})


In [ ]:
"""EMPOWER survey constants.

TIERS, KARMA_IZ_HB_HA, KMOS_BANDS, LINES_AA, OH_FORESTS_UM
    Selection windows and instrumental constants. Tier centres are taken from the
    EMPOWER proposal; ``dz`` is a "search half-width" used only for inventory
    counts — it is wider than the KARMA per-target window because we want to
    see what surveys deliver near each tier.

COSMO
    FlatLambdaCDM(H0=70, Om0=0.30) — used only for comoving-distance plots.

FIELD_CENTRES
    (RA, Dec) in degrees adopted for sky views; ASTRODEEP/MUSE coverage is
    smaller than the COSMOS box, so a wider 1.5 deg half-side is used for
    COSMOS only.

SPECZ_MATCH_RADII / BASE_MATCH_RADIUS
    Per-survey positional cross-match tolerances — the central knob for tuning
    matches to each survey's astrometric accuracy (see below).
"""

COSMO = FlatLambdaCDM(H0=70.0, Om0=0.30)

TIERS = {
    "T1 (z~0.75)": dict(zc=0.75, dz=0.10, color="#1f77b4"),
    "T2 (z~1.6)":  dict(zc=1.60, dz=0.20, color="#2ca02c"),
    "T3 (z~2.3)":  dict(zc=2.30, dz=0.20, color="#d62728"),
}

KARMA_IZ_HB_HA = (0.60, 0.6325)

KMOS_BANDS = {
    "IZ":  (0.78, 1.08),
    "YJ":  (1.025, 1.345),
    "H":   (1.460, 1.850),
    "HK":  (1.480, 2.450),
    "K":   (1.950, 2.450),
}

LINES_AA = {
    "H$\\beta$":     4861.33,
    "[OIII]5007":      5006.84,
    "H$\\alpha$":    6562.80,
    "[NII]6584":       6583.45,
}

# Approximate OH-forest envelopes (microns).  Used for visual context; for
# per-target sky-line filtering use a real OH sky spectrum.
OH_FORESTS_UM = [
    (0.86, 0.88), (0.92, 0.94), (1.08, 1.12), (1.18, 1.22),
    (1.27, 1.32), (1.42, 1.50), (1.55, 1.63), (1.73, 1.78),
    (1.83, 1.92),
]

FIELD_CENTRES = {
    "COSMOS":  dict(ra=150.116321, dec=+2.20583, half_deg=1.50),
    "GOODS-S": dict(ra= 53.123,    dec=-27.81,   half_deg=0.45),
}

# === Cross-match radii (arcsec), tunable PER SURVEY =========================
# The positional tolerance for calling two catalogue entries "the same source".
# Set per survey so it tracks each survey's astrometric accuracy: JWST NIRSpec
# (DJA, JADES) is tied to Gaia/NIRCam at sub-pixel level, ground-based slit/IFU
# surveys (VANDELS, MUSE) are looser, and AAOmega fibres (DEVILS) looser still.
# These radii drive BOTH (a) de-duplication of a survey against higher-priority
# layers in the unified compilation and (b) matching a survey onto a base
# catalogue (Khostovan, ASTRODEEP). Dial a value here when a survey's astrometry
# warrants it — every count downstream follows from this single dict.
SPECZ_MATCH_RADII = {
    "JADES-DR4": 0.5,   # JWST NIRSpec, target coords (Gaia/NIRCam-tied)
    "DJA":       0.5,   # JWST NIRSpec MSA, homogenised re-reduction
    "VANDELS":   0.75,  # VIMOS multi-slit, ground-based
    "MUSE-Wide": 0.75,  # MUSE IFU, emission-line centroid
    "MUSE-UDF":  0.75,  # MUSE IFU, emission-line centroid
    "DEVILS":    1.0,   # AAT/AAOmega 2" fibres, ground-based
}
BASE_MATCH_RADIUS = 1.0   # arcsec; fallback when a survey is absent from the dict

# GOODS-S spec-z layers in PRIORITY order (first wins in dedup) and their colours.
# JADES DR4 is placed ABOVE DJA so its rigorously-graded A/B/C redshifts override
# DJA's homogenised z_best wherever the two overlap.
GS_SURVEYS = ("JADES-DR4", "DJA", "VANDELS", "MUSE-Wide", "MUSE-UDF")
GS_PALETTE = {"JADES-DR4": "#9467bd", "DJA": "#1f77b4", "VANDELS": "#2ca02c",
              "MUSE-Wide": "#d62728", "MUSE-UDF": "#ff7f0e"}

# Output paths
HERE         = os.getcwd()
REPO_ROOT    = os.path.abspath(os.path.join(HERE, ".."))
SPEC_DIR     = os.path.join(REPO_ROOT, "catalogues")
FIG_DIR      = os.path.join(HERE, "figures", "specz_analysis")
os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
"""Helper functions used throughout."""

def tier_mask(z, tier_name):
    """Boolean array selecting redshifts inside the named EMPOWER tier window."""
    cfg = TIERS[tier_name]
    return (z >= cfg["zc"] - cfg["dz"]) & (z <= cfg["zc"] + cfg["dz"])


def line_obs_um(rest_aa, z):
    """Observed-frame wavelength (microns) for a rest-frame line at redshift z."""
    return (rest_aa / 1e4) * (1.0 + z)


@dataclass
class CrossmatchResult:
    """Result of a *positional* (blind) sky cross-match.

    Attributes
    ----------
    idx, sep_arcsec : np.ndarray
        Nearest-neighbour index into the reference catalogue and separation in
        arcsec, with the same length as the query catalogue.
    matched : np.ndarray of bool
        ``sep_arcsec < radius``.
    radius_arcsec : float
        The match radius used.
    note : str
        Always set to a warning string: these matches are NOT science-grade
        identifications.  Pairs flagged as catastrophic disagreements should
        be vetted manually before use.
    """
    idx:           np.ndarray
    sep_arcsec:    np.ndarray
    matched:       np.ndarray
    radius_arcsec: float
    note:          str = (
        "BLIND-MATCH CAVEAT: nearest-neighbour positional match only — use to "
        "count gaps, not to confirm individual source identities. Vet "
        "catastrophic-z disagreements with image-matched IDs offline."
    )


def blind_xmatch(ra_query, dec_query, ra_ref, dec_ref, radius_arcsec=1.0):
    """Positional 1-arcsec sky cross-match returning a :class:`CrossmatchResult`."""
    c_q = SkyCoord(ra_query, dec_query, unit="deg")
    c_r = SkyCoord(ra_ref,   dec_ref,   unit="deg")
    idx, sep, _ = c_q.match_to_catalog_sky(c_r)
    arcsec = sep.arcsec
    return CrossmatchResult(
        idx=np.asarray(idx), sep_arcsec=np.asarray(arcsec),
        matched=arcsec < radius_arcsec, radius_arcsec=radius_arcsec,
    )


def shade_tier_windows(ax, transform=None, label=True):
    """Overlay the three EMPOWER tier windows and the KARMA IZ Hβ+Hα window."""
    if transform is None:
        transform = ax.get_xaxis_transform()
    for name, cfg in TIERS.items():
        ax.axvspan(cfg["zc"] - cfg["dz"], cfg["zc"] + cfg["dz"],
                   color=cfg["color"], alpha=0.08, zorder=-1)
        ax.axvline(cfg["zc"], color=cfg["color"], lw=0.7, ls="--", alpha=0.55)
        if label:
            ax.text(cfg["zc"], 0.97, name, transform=transform,
                    ha="center", va="top", fontsize=8, color=cfg["color"])
    ax.axvspan(*KARMA_IZ_HB_HA, color="navy", alpha=0.20, zorder=-1)


def save_fig(fig, name):
    """Save under FIG_DIR/<name>.png and report path."""
    p = os.path.join(FIG_DIR, name + ".png")
    fig.savefig(p)
    return p


def match_radius(survey):
    """Per-survey positional match radius (arcsec); see SPECZ_MATCH_RADII."""
    return SPECZ_MATCH_RADII.get(survey, BASE_MATCH_RADIUS)


### 1.1 Catalogue loaders

Catalogue locations and a single loader per file. Each loader returns an
``astropy.table.Table`` and prints one line describing what was read. The DJA
NIRSpec compilation is fetched from Zenodo and cached locally by ``astropy``.


In [ ]:
"""Catalogue loaders. Each returns an `astropy.table.Table`."""
import shutil
from astropy.utils.data import download_file

DJA_URL = ("https://zenodo.org/records/15472354/files/"
           "dja_msaexp_emission_lines_v4.4.csv.gz")


def _path(*parts):
    return os.path.join(SPEC_DIR, *parts)


def load_dja_nirspec():
    """DJA NIRSpec v4.4 (Brammer / Heintz, Zenodo 15472354).

    Reads the local copy in catalogues/dja_v4.4/ (see _provenance.txt). The
    133 MB table is too large to track in git, so on a fresh checkout (e.g.
    running from the web) it is fetched from Zenodo into that folder on first
    use; subsequent runs read it from disk.
    """
    p = _path("dja_v4.4", "dja_msaexp_emission_lines_v4.4.csv.gz")
    if not os.path.exists(p):
        os.makedirs(os.path.dirname(p), exist_ok=True)
        shutil.copy(download_file(DJA_URL, cache=True), p)
    return Table.read(p, format="csv")


def load_khostovan_cosmos():
    """Khostovan et al. 2026 (ApJS 282, 6) COSMOS spec-z compilation, unique table."""
    p = _path("cosmos_khostovan25", "specz_compilation_COSMOS_DR1.1_unique.fits")
    return Table.read(p)


def load_devils_d10():
    """DEVILS DR1 D10 (Davies et al. 2025) — rows with a real DEVILS spec-z."""
    return Table.read(_path("devils_dr1", "devils_dr1_d10_devils_specz.fits"))


def load_astrodeep_phys():
    """ASTRODEEP-GS43 (Merlin+21) physical-properties table."""
    return Table.read(_path("astrodeep_gs43", "astrodeep_gs43_phys.fits"))


def load_astrodeep_phot():
    """ASTRODEEP-GS43 (Merlin+21) photometric table (carries RA/Dec)."""
    return Table.read(_path("astrodeep_gs43", "astrodeep_gs43_phot.fits"))


def load_muse_wide():
    """MUSE-Wide DR1 emission-line objects (Urrutia+19)."""
    return Table.read(_path("muse_wide_dr1", "muse_wide_dr1_emobj.fits"))


def load_muse_udf():
    """AMUSED MUSE-UDF DR2 main (Bacon+23)."""
    return Table.read(_path("amused_museudf_dr2", "amused_museudf_dr2_dr2_main.fits"))


def load_vandels_cdfs():
    """VANDELS DR4 CDFS spectra (Garilli+21)."""
    return Table.read(_path("vandels_dr4_cdfs", "vandels_dr4_cdfs_cdfs.fits"))


def load_jades_dr4():
    """JADES DR4 NIRSpec combined catalogue (Curtis-Lake+25 / Scholtz+25), HDU 1.

    Returns the 5,190-row 'Obs_info' target table. Best redshift is ``z_Spec``
    with byte-string quality flag ``z_Spec_flag`` ('A'..'E'); ``Field`` is
    b"GS"/b"GN". Pulled from https://jades.herts.ac.uk/DR4/ (not yet on VizieR).
    """
    return Table.read(_path("jades_dr4", "Combined_DR4_external_v1.2.1.fits"), hdu=1)


## 2 · COSMOS

### 2.1 Khostovan+25 baseline

Khostovan et al. 2026 (ApJS 282, 6) compile ≈488 000 spectroscopic redshifts
covering the 2 deg² COSMOS field. The *unique* table reduces duplicates by
keeping the entry with the highest ``flag`` per source (flag = 4: highest
confidence, 3: secure, 2: probable, 1: uncertain, 0: unknown, 14/19: stellar).
The columns we use are ``ra_corrected``, ``dec_corrected``, ``specz``, ``flag``
and ``survey`` (numeric ID listed in ``..._surveys.list``).


In [ ]:
"""Load the Khostovan+25 baseline and report basic statistics."""
khostovan = load_khostovan_cosmos()

z   = np.asarray(khostovan["specz"])
flag = np.asarray(khostovan["flag"])

is_secure = np.isin(flag, [3, 4])
is_top    = (flag == 4)

print(f"Khostovan DR1.1 unique rows      : {len(khostovan):,}")
print(f"  flag = 4 (highest)             : {is_top.sum():,}")
print(f"  flag ≥ 3 (secure)              : {is_secure.sum():,}")
print(f"  z in tier 1 (0.65–0.85), secure: {(is_secure & tier_mask(z, 'T1 (z~0.75)')).sum():,}")
print(f"  z in tier 2 (1.4–1.8),   secure: {(is_secure & tier_mask(z, 'T2 (z~1.6)')).sum():,}")
print(f"  z in tier 3 (2.1–2.5),   secure: {(is_secure & tier_mask(z, 'T3 (z~2.3)')).sum():,}")
print(f"  z in KARMA IZ Hβ+Hα window     : "
      f"{(is_secure & (z>KARMA_IZ_HB_HA[0]) & (z<KARMA_IZ_HB_HA[1])).sum():,}")


In [ ]:
"""Figure 1 — Khostovan z-distribution with EMPOWER tier overlays.

Shows the spec-z budget Khostovan provides per EMPOWER tier window. Note the
KARMA IZ Hβ+Hα slice (navy band) carries roughly 3 700 secure sources, but it
is only 0.03 wide in z.
"""
fig, ax = plt.subplots(figsize=(10.5, 4))
bins = np.arange(0.0, 4.0, 0.02)

ax.hist(z[(z > 0) & ~np.isin(flag, [14, 19])], bins=bins,
        histtype="stepfilled", color="0.86", edgecolor="0.65",
        label=f"all valid (N={((z>0)&~np.isin(flag,[14,19])).sum():,})")
ax.hist(z[is_secure & (z > 0)], bins=bins, histtype="step",
        color="0.20", lw=1.4, label=f"flag ≥ 3  (N={(is_secure & (z>0)).sum():,})")
ax.hist(z[is_top & (z > 0)], bins=bins, histtype="stepfilled",
        color="#ff8a3d", alpha=0.65, edgecolor="#d2691e",
        label=f"flag = 4 (N={(is_top & (z>0)).sum():,})")

shade_tier_windows(ax)
ax.set_yscale("log")
ax.set_xlim(0, 3.5)
ax.set_xlabel("Spectroscopic redshift")
ax.set_ylabel("N (per Δz = 0.02)")
ax.set_title("COSMOS — Khostovan+25 spec-z by quality flag, with EMPOWER tier windows")
ax.legend(loc="upper right", frameon=True)
fig.tight_layout()
save_fig(fig, "fig01_cosmos_khostovan_zhist")
plt.show()


In [ ]:
"""Figure 2 — Per-tier sky maps for COSMOS flag = 4.

Each panel restricts Khostovan flag = 4 to one tier window. The leftmost panel
also overlays the KARMA IZ Hβ+Hα subset (cyan crosses), and the locations of
the 24 IFUs actually observed in COSMOS in February 2026 (gold stars; loaded
from the EMPOWER curated dataset).
"""
ra  = np.asarray(khostovan["ra_corrected"])
dec = np.asarray(khostovan["dec_corrected"])

# Read the 24 observed IFUs (small file; in the same Dropbox tree)
try:
    kmos_obs = Table.read(_path("drive-download-20260515T072221Z-3-001", "COSMOS",
                                "observed targets",
                                "cosmos_observed_11_16_02_2026_target.fits"))
except FileNotFoundError:
    kmos_obs = None

cc = FIELD_CENTRES["COSMOS"]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.6), sharey=True)
for ax, name in zip(axes, TIERS):
    cfg = TIERS[name]
    m = is_top & tier_mask(z, name)
    ax.scatter(ra[m], dec[m], s=3, c=cfg["color"], alpha=0.55,
               edgecolors="none", label=f"flag=4  (N={m.sum():,})")
    if name == "T1 (z~0.75)":
        karma = is_top & (z > KARMA_IZ_HB_HA[0]) & (z < KARMA_IZ_HB_HA[1])
        ax.scatter(ra[karma], dec[karma], s=12, marker="x",
                   c="navy", lw=0.5, label=f"IZ Hβ+Hα ({karma.sum():,})")
        if kmos_obs is not None:
            ax.scatter(kmos_obs["ra"], kmos_obs["dec"], s=70, marker="*",
                       facecolors="gold", edgecolors="k", lw=0.4,
                       label="KMOS Feb 2026")
    ax.set_aspect("equal")
    ax.set_xlim(cc["ra"] + cc["half_deg"], cc["ra"] - cc["half_deg"])
    ax.set_ylim(cc["dec"] - cc["half_deg"], cc["dec"] + cc["half_deg"])
    ax.set_xlabel("RA [deg]")
    ax.set_title(name)
    ax.legend(loc="upper left", fontsize=7)
axes[0].set_ylabel("Dec [deg]")
fig.suptitle("COSMOS — Khostovan flag = 4 sky distribution per EMPOWER tier", y=1.02)
fig.tight_layout()
save_fig(fig, "fig02_cosmos_sky_per_tier")
plt.show()


In [ ]:
"""Figure 3 — COSMOS large-scale-structure wedge (RA–z, Dec–z).

Plots Khostovan flag = 4 in two projections to make large-scale-structure
features (e.g. the z ≈ 0.73 COSMOS Wall) visible to the reader.
"""
m = is_top & (z > 0) & (z < 3.0)
fig, axes = plt.subplots(2, 1, figsize=(11.5, 6.3), sharex=True)
for ax, yarr, lbl in zip(axes, [ra, dec], ["RA [deg]", "Dec [deg]"]):
    ax.scatter(z[m], yarr[m], s=1.4, c="0.25", alpha=0.35, edgecolors="none")
    shade_tier_windows(ax, label=False)
    ax.set_ylabel(lbl)
axes[-1].set_xlabel("Spectroscopic redshift")
axes[0].set_title("COSMOS large-scale structure (flag = 4)")
axes[0].set_xlim(0, 3.0)
fig.tight_layout()
save_fig(fig, "fig03_cosmos_wedge_lss")
plt.show()


In [ ]:
"""Figure 3b — 3D comoving view of the z≈0.73 COSMOS Wall."""
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

cc_cos = FIELD_CENTRES["COSMOS"]
t1_cfg = TIERS["T1 (z~0.75)"]

valid_top = is_top & np.isfinite(z) & (z > 0)
m_t1 = valid_top & (z >= t1_cfg["zc"] - t1_cfg["dz"]) & (z <= t1_cfg["zc"] + t1_cfg["dz"])

z_wall = 0.73
dz_wall = 0.02  # widen/narrow this to change the highlighted wall slice
m_wall = m_t1 & (np.abs(z - z_wall) <= dz_wall)

chi_ref = COSMO.comoving_distance(z_wall).to_value(u.Mpc)
dec0_rad = np.deg2rad(cc_cos["dec"])

# Small-angle local comoving coordinates around the COSMOS field centre
x_t1 = chi_ref * np.deg2rad(ra[m_t1] - cc_cos["ra"]) * np.cos(dec0_rad)
y_t1 = chi_ref * np.deg2rad(dec[m_t1] - cc_cos["dec"])
dchi_t1 = COSMO.comoving_distance(z[m_t1]).to_value(u.Mpc) - chi_ref

x_wall = chi_ref * np.deg2rad(ra[m_wall] - cc_cos["ra"]) * np.cos(dec0_rad)
y_wall = chi_ref * np.deg2rad(dec[m_wall] - cc_cos["dec"])
dchi_wall = COSMO.comoving_distance(z[m_wall]).to_value(u.Mpc) - chi_ref

fig_wall = plt.figure(figsize=(9.4, 7.4))
ax_wall = fig_wall.add_subplot(111, projection="3d")

ax_wall.scatter(
    x_t1, y_t1, dchi_t1,
    s=1.5, c="0.78", alpha=0.10, edgecolors="none",
    label=f"T1 flag=4 (N={m_t1.sum():,})"
)
sc_wall = ax_wall.scatter(
    x_wall, y_wall, dchi_wall,
    s=12, c=z[m_wall], cmap="viridis", alpha=0.95, edgecolors="none",
    label=f"Wall slice ({z_wall-dz_wall:.2f} < z < {z_wall+dz_wall:.2f}; N={m_wall.sum():,})"
)

# Reference plane at z = 0.73
xx, yy = np.meshgrid(np.linspace(x_t1.min(), x_t1.max(), 2),
                     np.linspace(y_t1.min(), y_t1.max(), 2))
ax_wall.plot_surface(xx, yy, np.zeros_like(xx), color="navy", alpha=0.04, linewidth=0)

ax_wall.set_xlabel(r"$\Delta x_{\rm RA}$ [comoving Mpc]")
ax_wall.set_ylabel(r"$\Delta y_{\rm Dec}$ [comoving Mpc]")
ax_wall.set_zlabel(r"$\Delta \chi$ from $z=0.73$ [Mpc]")
ax_wall.set_title("COSMOS Wall near z≈0.73 — 3D comoving view (Khostovan flag = 4)")
ax_wall.view_init(elev=18, azim=35)

if len(x_t1) > 1:
    ax_wall.set_box_aspect((np.ptp(x_t1), np.ptp(y_t1), np.ptp(dchi_t1)))

cbar = fig_wall.colorbar(sc_wall, ax=ax_wall, pad=0.08, shrink=0.82)
cbar.set_label("Spectroscopic redshift")

ax_wall.legend(loc="upper left", fontsize=8)
fig_wall.tight_layout()

out_path = os.path.join(FIG_DIR, "fig03b_cosmos_wall_3d.png")
fig_wall.savefig(out_path, dpi=150, bbox_inches="tight")
# plt.show()



### 2.2 COSMOS — what is missing from Khostovan?

Two recent surveys cover material the Khostovan compilation does not yet
absorb:

* **DJA NIRSpec v4.4** (Brammer / Heintz) — re-reduced JWST NIRSpec MSA / IFU
  data, JADES-dominated. Provides high-S/N grating redshifts for thousands of
  faint compact targets.
* **DEVILS DR1 D10** (Davies+25) — AAT/AAOmega Y < 21.2 survey covering the
  central ≈ 1.5 deg² of COSMOS.

The cross-match below is positional (1 arcsec) — see the caveat at the top of
the notebook.


In [ ]:
"""Filter DJA NIRSpec v4.4 to the COSMOS box and report grade composition."""
tab_dja = load_dja_nirspec()
cc = FIELD_CENTRES["COSMOS"]
in_cos = ((tab_dja["ra"]  > cc["ra"]  - cc["half_deg"]) &
          (tab_dja["ra"]  < cc["ra"]  + cc["half_deg"]) &
          (tab_dja["dec"] > cc["dec"] - cc["half_deg"]) &
          (tab_dja["dec"] < cc["dec"] + cc["half_deg"]))
dja_cos = tab_dja[in_cos]
print(f"DJA inside COSMOS box ({cc['half_deg']*2}° square): {len(dja_cos):,}")
for g in (3, 2, 1, 0):
    print(f"  grade = {g}: {(dja_cos['grade'] == g).sum():,}")
print(f"  grade other / blank : "
      f"{(~np.isin(dja_cos['grade'], [0, 1, 2, 3])).sum():,}")


In [ ]:
"""Cross-match DJA -> Khostovan and classify each DJA source.

The match radius is DJA-specific (SPECZ_MATCH_RADII["DJA"]), reflecting JWST
NIRSpec's sub-pixel astrometry rather than a blanket 1".

Categories:
    matched  : DJA source has *any* Khostovan entry within the DJA radius
    new      : DJA source has no Khostovan counterpart within the DJA radius
    upgrade  : matched, Khostovan entry has flag < 3 (low / no confidence)
"""
r_dja = match_radius("DJA")
xm_dja = blind_xmatch(
    dja_cos["ra"], dja_cos["dec"], ra, dec, radius_arcsec=r_dja,
)
khost_flag_d = flag[xm_dja.idx]
khost_z_d    = z[xm_dja.idx]
matched      = xm_dja.matched
secure_d     = (dja_cos["grade"] == 3) & (dja_cos["z_best"] > 0)

new_to_base  = secure_d & ~matched
upgrade      = secure_d & matched & ~np.isin(khost_flag_d, [3, 4])
agree_pool   = secure_d & matched &  np.isin(khost_flag_d, [3, 4])

print(f"DJA grade=3 sources in COSMOS  : {secure_d.sum():,}  "
      f"(match radius = {r_dja}\u2033)")
print(f"  NEW   (no Khostovan match)   : {new_to_base.sum():>5,}")
print(f"  UPGRADE (Khostovan flag < 3) : {upgrade.sum():>5,}")
print(f"  agree-test pool (flag \u2265 3)   : {agree_pool.sum():>5,}")
print()
print(xm_dja.note)


In [ ]:
"""Figure 4 — COSMOS NIRSpec gap analysis.

Top: per-grade DJA stack split into "in Khostovan @ 1″" vs missing. Bottom:
how the catastrophic z-disagreement rate depends on the Khostovan flag of the
matched entry (i.e. the chance-match contaminant rate).
"""
DZ_THR = 0.01

# Agreement against Khostovan, restricted to DJA grade=3 and Khostovan z>0
both = secure_d & matched & (khost_z_d > 0) & np.isfinite(dja_cos["z_best"])
dz   = ((dja_cos["z_best"][both] - khost_z_d[both]) /
        (1.0 + khost_z_d[both]))
flag_b = khost_flag_d[both]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# (a) per-grade stack
ax = axes[0]
grades = [3, 2, 1]
n_in    = np.array([((dja_cos["grade"] == g) &  matched).sum() for g in grades])
n_miss  = np.array([((dja_cos["grade"] == g) & ~matched).sum() for g in grades])
xs = np.arange(len(grades))
ax.bar(xs, n_in,   color="#1f77b4", label="in Khostovan (1″)")
ax.bar(xs, n_miss, bottom=n_in, color="#d62728", label="missing")
for xi, (a, b) in enumerate(zip(n_in, n_miss)):
    ax.text(xi, a + b, f"{a:,}+{b:,}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs)
ax.set_xticklabels([f"DJA gr {g}" for g in grades])
ax.set_ylabel("N (in COSMOS box)")
ax.set_title("(a) DJA sources — present / missing in Khostovan")
ax.legend(fontsize=8)

# (b) catastrophic-z rate by Khostovan flag
ax = axes[1]
flags_plot = [4, 3, 2, 1, 0]
n_pairs = [(flag_b == f).sum() for f in flags_plot]
n_out   = [((flag_b == f) & (np.abs(dz) > DZ_THR)).sum() for f in flags_plot]
n_ok    = [n_pairs[i] - n_out[i] for i in range(len(flags_plot))]
ax.bar(range(len(flags_plot)), n_ok,  color="#1f77b4", label="|Δz/(1+z)|≤0.01")
ax.bar(range(len(flags_plot)), n_out, bottom=n_ok, color="#d62728",
       label="catastrophic")
for i in range(len(flags_plot)):
    tot = n_pairs[i]
    if tot > 0:
        ax.text(i, tot, f"{n_out[i]/tot*100:.0f}%",
                ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(flags_plot)))
ax.set_xticklabels([str(f) for f in flags_plot])
ax.set_xlabel("Khostovan flag of matched entry")
ax.set_ylabel("N matched DJA grade=3 pairs")
ax.set_title("(b) catastrophic-z rate by Khostovan flag")
ax.legend(fontsize=8)

fig.tight_layout()
save_fig(fig, "fig04_cosmos_dja_gap")
plt.show()

print()
print("Catastrophic disagreement among Khostovan-secure (flag≥3) matches:",
      f"{(((flag_b>=3) & (np.abs(dz)>DZ_THR)).sum())} / {(flag_b>=3).sum()} "
      f"= {((flag_b>=3) & (np.abs(dz)>DZ_THR)).sum() / max((flag_b>=3).sum(),1):.0%}")
print("Pairs flagged here are CANDIDATES for manual review (see notebook caveat).")


In [ ]:
"""Cross-match DEVILS DR1 D10 -> Khostovan and report new-vs-known.

DEVILS is shallower (Y < 21.2) and overlaps the central COSMOS sub-region only.
Its main contribution is at z < 1 — most overlap with Khostovan is expected.
The match radius is DEVILS-specific (AAOmega 2" fibres -> looser tolerance).
"""
devils = load_devils_d10()
clean = devils[(devils["starFlag"] == 0)
               & (devils["artefactFlag"] == 0)
               & (devils["mask"] == 0)
               & (devils["DEVILS_z"] > 0)]
r_dev = match_radius("DEVILS")
xm_dev = blind_xmatch(
    clean["RAcen"], clean["DECcen"], ra, dec, radius_arcsec=r_dev,
)
m_sec = clean["DEVILS_prob"] >= 0.8

print(f"DEVILS clean spec-z              : {len(clean):,}  "
      f"(match radius = {r_dev}\u2033)")
print(f"  DEVILS_prob \u2265 0.8              : {m_sec.sum():,}")
print(f"     NEW to Khostovan            : {(m_sec & ~xm_dev.matched).sum():,}")
print(f"     in Khostovan                : {(m_sec &  xm_dev.matched).sum():,}")
print()
print("Per-tier DEVILS_prob\u22650.8 sources NEW to Khostovan:")
for name in TIERS:
    n_new = (m_sec & ~xm_dev.matched & tier_mask(clean["DEVILS_z"], name)).sum()
    print(f"  {name:<14s}{n_new:>5,}")
print()
print(xm_dev.note)


In [ ]:
"""Figure 5 — Final COSMOS per-tier inventory.

For each tier we plot (a) the number of secure (flag≥3) Khostovan sources, (b)
how many DJA grade=3 sources are new to Khostovan, (c) how many DEVILS
prob≥0.8 sources are new to Khostovan. Targets in (b) and (c) are candidates
to add to the EMPOWER selection pool *after* cross-matching them to the
COSMOS photometric reference (e.g. COSMOS2020) for stellar masses, which is
left to a dedicated follow-up notebook.
"""
fig, ax = plt.subplots(figsize=(8.5, 4.2))
names = list(TIERS.keys()) + ["KARMA IZ Hβ+Hα"]
khostovan_in = []
dja_new      = []
devils_new   = []
for name in TIERS:
    khostovan_in.append((is_secure & tier_mask(z, name)).sum())
    dja_new.append((secure_d & ~matched & tier_mask(dja_cos["z_best"], name)).sum())
    devils_new.append((m_sec & ~xm_dev.matched &
                       tier_mask(clean["DEVILS_z"], name)).sum())

# KARMA window
khost_k  = (is_secure & (z > KARMA_IZ_HB_HA[0]) & (z < KARMA_IZ_HB_HA[1])).sum()
dja_k    = (secure_d & ~matched
            & (dja_cos["z_best"] > KARMA_IZ_HB_HA[0])
            & (dja_cos["z_best"] < KARMA_IZ_HB_HA[1])).sum()
devils_k = (m_sec & ~xm_dev.matched
            & (clean["DEVILS_z"] > KARMA_IZ_HB_HA[0])
            & (clean["DEVILS_z"] < KARMA_IZ_HB_HA[1])).sum()
khostovan_in.append(khost_k)
dja_new.append(dja_k)
devils_new.append(devils_k)

xs = np.arange(len(names))
w = 0.27
# Use linear scale and put DJA/DEVILS on a twin axis since their counts are tiny
# compared with Khostovan. That keeps the figure readable.
ax2 = ax.twinx()
b0 = ax.bar(xs - w, khostovan_in, w, color="#1f77b4",
            label="Khostovan flag≥3", alpha=0.85)
b1 = ax2.bar(xs,    dja_new,      w, color="#2ca02c",
             label="DJA grade=3 NEW", alpha=0.85)
b2 = ax2.bar(xs + w, devils_new,  w, color="#d62728",
             label="DEVILS prob≥0.8 NEW", alpha=0.85)

for i, n in enumerate(khostovan_in):
    if n > 0:
        ax.text(xs[i]-w, n, f"{n:,}", ha="center", va="bottom", fontsize=7.5)
for i, n in enumerate(dja_new):
    if n > 0:
        ax2.text(xs[i], n, f"{n}", ha="center", va="bottom", fontsize=7.5)
for i, n in enumerate(devils_new):
    if n > 0:
        ax2.text(xs[i]+w, n, f"{n}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(xs)
ax.set_xticklabels(names)
ax.set_ylabel("N (Khostovan flag ≥ 3)", color="#1f77b4")
ax2.set_ylabel("N (DJA / DEVILS NEW to Khostovan)", color="0.20")
ax.tick_params(axis="y", colors="#1f77b4")
ax.set_title("COSMOS — spec-z budget per EMPOWER tier "
             "(baseline + add-on contributions)")
handles = [b0, b1, b2]
labels  = [h.get_label() for h in handles]
ax.legend(handles, labels, loc="upper right", fontsize=8)
fig.tight_layout()
save_fig(fig, "fig05_cosmos_summary")
plt.show()


## 3 · GOODS-S

### 3.1 ASTRODEEP-GS43 baseline — and why it is not a spec-z catalogue

Merlin et al. 2021 (A&A 649, A22) is the most complete public photometric +
photometric-redshift catalogue for CANDELS GOODS-S (35,108 sources, 43-band
photometry). Per the published documentation, ``zBest`` is "the best photo-z
estimate", i.e. the median of the three photo-z estimators (``zLePhare``,
``zEAzY``, ``zz_phot``). The ``zspec_survey`` column carries only a *tag*
identifying which spec-z survey has observed each source — the **actual prior
spec-z value is not stored**. We verified this empirically with DJA: matched
DJA grade=3 ↔ ASTRODEEP pairs differ by median Δz/(1+z) ≈ +0.46 with
NMAD ≈ 0.76, which is photo-z scatter, not inter-survey disagreement.

ASTRODEEP-GS43 is therefore used in this notebook as **the photometric and
mass reference**, not as a spec-z compilation. The spec-z compilation for
GOODS-S is built explicitly in §3.2 by stacking DJA + VANDELS + MUSE-Wide +
MUSE-UDF.


In [ ]:
"""Load ASTRODEEP-GS43 photometry + physical-properties tables and build the
working `astrodeep` view (RA, Dec, zBest, log M*, spec-z tag flag)."""
gs43_phot = load_astrodeep_phot()
gs43_phys = load_astrodeep_phys()

astrodeep = Table()
astrodeep["RA"]      = gs43_phot["RAJ2000"]
astrodeep["Dec"]     = gs43_phot["DEJ2000"]
astrodeep["zphot"]   = gs43_phys["zBest"]            # photo-z, NOT spec-z
astrodeep["has_tag"] = np.array(
    [str(s).strip() != "-" for s in gs43_phys["zspec_survey"]]
)
with np.errstate(divide="ignore", invalid="ignore"):
    astrodeep["logM"] = np.log10(np.asarray(gs43_phys["M_tau"]) * 1e9)

print(f"ASTRODEEP-GS43 sources  : {len(astrodeep):,}")
print(f"  with spec-z tag       : {astrodeep['has_tag'].sum():,}")
print(f"  M* ≥ 1e10             : {(astrodeep['logM'] >= 10).sum():,}")
print(f"  M* ≥ 1e10 and tagged  : "
      f"{((astrodeep['logM'] >= 10) & astrodeep['has_tag']).sum():,}")


### 3.2 Building the GOODS-S spec-z compilation

We stack **five** secure-z layers and reduce them to one entry per physical
source with a greedy, **per-survey-radius** dedup (see `dedup_layers`), using the
priority order

> **JADES DR4 (A/B/C) ▶ DJA grade=3 ▶ VANDELS q_zsp≥3 ▶ MUSE-Wide Conf≥2 ▶ MUSE-UDF DR2**.

JADES DR4 is placed first so its rigorously-graded NIRSpec redshifts win over
DJA's homogenised `z_best` wherever they overlap; DJA comes next (JWST grating >
deep optical slit > IFS). The match radius is set **per survey** in
`SPECZ_MATCH_RADII` (0.5″ for JWST NIRSpec, 0.75″ for VIMOS/MUSE, 1.0″ for
AAOmega), so each layer dedups at its own astrometric accuracy.

The greedy pass also collapses **intra-survey** duplicates — important because
DJA NIRSpec is per-spectrum (multiple gratings/visits per object), so ~8,800 of
its 9,647 GOODS-S grade=3 rows are repeats of the same source. The output
`gs_specz` table is the unique-source GOODS-S spec-z reference used below.


In [ ]:
"""Load the five spec-z layers and report per-survey counts."""
dja_gs   = tab_dja[((tab_dja["ra"]  > 52.8) & (tab_dja["ra"]  < 53.5) &
                    (tab_dja["dec"] > -28.1) & (tab_dja["dec"] < -27.5))]
muse_wide = load_muse_wide()
muse_udf  = load_muse_udf()
vandels   = load_vandels_cdfs()
jades_dr4 = load_jades_dr4()

# JADES DR4 carries byte-string columns — decode before comparing.
def _decode(col):
    return np.array([s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
                     for s in np.asarray(col)])
dr4_flag  = _decode(jades_dr4["z_Spec_flag"])
dr4_field = _decode(jades_dr4["Field"])
dr4_z     = np.asarray(jades_dr4["z_Spec"], dtype=float)
# GOODS-S, robust (A/B/C), valid redshift
dr4_gs_abc = ((dr4_field == "GS") & np.isin(dr4_flag, ["A", "B", "C"])
              & np.isfinite(dr4_z) & (dr4_z > 0))

print(f"JADES DR4 (combined)    : {len(jades_dr4):,}  "
      f"(GOODS-S A/B/C: {dr4_gs_abc.sum():,})")
print(f"DJA (GOODS-S box)       : {len(dja_gs):,}  "
      f"(grade=3: {((dja_gs['grade'] == 3) & (dja_gs['z_best'] > 0)).sum():,})")
print(f"MUSE-Wide DR1           : {len(muse_wide):,}  "
      f"(Conf\u22652: {(muse_wide['Conf']  >= 2).sum():,})")
print(f"AMUSED MUSE-UDF DR2     : {len(muse_udf):,}  "
      f"(z>0: {(muse_udf['z']  > 0).sum():,})")
print(f"VANDELS DR4 CDFS        : {len(vandels):,}  "
      f"(q_zsp\u22653: {(vandels['q_zsp'] >= 3).sum():,})")


In [ ]:
"""Build the unified GOODS-S spec-z table with priority-ordered, per-survey dedup.

Layers are stacked in GS_SURVEYS priority order (JADES-DR4 first, then DJA, then
the optical/IFU surveys) and reduced to ONE entry per physical source by a greedy
pass: walking the stacked rows in priority order, a row is kept only if no
already-kept row sits within *its own* survey's match radius (SPECZ_MATCH_RADII).
This collapses BOTH cross-survey duplicates AND intra-survey duplicates — the
latter matters because DJA NIRSpec is per-spectrum (multiple gratings/visits per
source), so ~8,800 of its 9,647 grade=3 rows are repeat observations of the same
object. JWST layers dedup tightly (0.5") and ground-based layers looser.
"""

def _make_layer(ra, dec, z, qual, survey):
    """Build a Table with the standard schema for one spec-z layer."""
    out = Table()
    out["ra"]     = np.asarray(ra,   dtype=float)
    out["dec"]    = np.asarray(dec,  dtype=float)
    out["z"]      = np.asarray(z,    dtype=float)
    out["qual"]   = np.asarray(qual, dtype=float)
    out["survey"] = np.array([survey] * len(out), dtype="U16")
    return out


def dedup_layers(layers, priority, radii):
    """Greedy unique-source dedup over layers stacked in priority order.

    A row j is kept iff no earlier (= higher-priority, or same-survey-earlier)
    KEPT row lies within ``radii[survey_j]`` of it. Implemented with a KD-tree on
    a local tangent-plane projection (arcsec); valid because GOODS-S spans < 0.5
    deg so planar distance ~ angular separation.
    """
    from scipy.spatial import cKDTree
    stacked = vstack([layers[k] for k in priority], join_type="outer")
    ra  = np.asarray(stacked["ra"],  dtype=float)
    dec = np.asarray(stacked["dec"], dtype=float)
    surv = np.asarray(stacked["survey"])
    dec0 = float(np.median(dec))
    xy = np.column_stack([ra * np.cos(np.radians(dec0)) * 3600.0, dec * 3600.0])
    tree = cKDTree(xy)
    rad = np.array([radii.get(s, BASE_MATCH_RADIUS) for s in surv])
    keep = np.zeros(len(stacked), dtype=bool)
    for j in range(len(stacked)):
        neigh = tree.query_ball_point(xy[j], rad[j])
        if any((k < j) and keep[k] for k in neigh):
            continue
        keep[j] = True
    return stacked[keep]


# Per-survey secure-redshift masks
sec_dja = (dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)
sec_van = (vandels["q_zsp"] >= 3) & (vandels["zsp"] > 0)
sec_mw  = (muse_wide["Conf"] >= 2) & (muse_wide["z"] > 0)
sec_mu  = muse_udf["z"] > 0

# JADES DR4 quality A/B/C -> numeric qual 3/2/1
_qmap    = {"A": 3.0, "B": 2.0, "C": 1.0}
dr4_qual = np.array([_qmap.get(f, 0.0) for f in dr4_flag])

gs_layers = {
    "JADES-DR4": _make_layer(jades_dr4["RA_TARG"][dr4_gs_abc],
                             jades_dr4["Dec_TARG"][dr4_gs_abc],
                             dr4_z[dr4_gs_abc], dr4_qual[dr4_gs_abc], "JADES-DR4"),
    "DJA":       _make_layer(dja_gs["ra"][sec_dja], dja_gs["dec"][sec_dja],
                             dja_gs["z_best"][sec_dja],
                             np.full(sec_dja.sum(), 3), "DJA"),
    "VANDELS":   _make_layer(vandels["RAJ2000"][sec_van], vandels["DEJ2000"][sec_van],
                             vandels["zsp"][sec_van], vandels["q_zsp"][sec_van], "VANDELS"),
    "MUSE-Wide": _make_layer(muse_wide["RAJ2000"][sec_mw], muse_wide["DEJ2000"][sec_mw],
                             muse_wide["z"][sec_mw], muse_wide["Conf"][sec_mw], "MUSE-Wide"),
    "MUSE-UDF":  _make_layer(muse_udf["RAJ2000"][sec_mu], muse_udf["DEJ2000"][sec_mu],
                             muse_udf["z"][sec_mu], np.full(sec_mu.sum(), 3), "MUSE-UDF"),
}

gs_specz = dedup_layers(gs_layers, GS_SURVEYS, SPECZ_MATCH_RADII)
print(f"Unified GOODS-S spec-z (per-survey dedup): {len(gs_specz):,}")
for s in GS_SURVEYS:
    print(f"  {s:<10s} {(gs_specz['survey'] == s).sum():>5,}  "
          f"(dedup r = {SPECZ_MATCH_RADII[s]}\u2033)")


In [ ]:
"""Figure 6 — GOODS-S unified spec-z sky map + redshift distribution.

Panel (a): per-survey sky distribution over the ASTRODEEP photometric source list.
Panel (b): per-survey redshift distribution. The complementary roles of the five
layers are visible: MUSE-Wide/UDF dominate at z < 1 (and inside the KARMA window),
while JADES DR4 / DJA dominate at z > 1.5.
"""
fig = plt.figure(figsize=(13.5, 5.4))
gs   = gridspec.GridSpec(1, 2, width_ratios=[1.0, 1.4])
palette = GS_PALETTE

# (a) sky
ax = fig.add_subplot(gs[0, 0])
ax.scatter(astrodeep["RA"], astrodeep["Dec"], s=0.6, c="0.88",
           alpha=0.4, edgecolors="none", label=f"ASTRODEEP (N={len(astrodeep):,})")
for s in GS_SURVEYS:
    m = gs_specz["survey"] == s
    ax.scatter(gs_specz["ra"][m], gs_specz["dec"][m], s=3, c=palette[s], alpha=0.7,
               edgecolors="none", label=f"{s} (N={m.sum():,})")
ax.set_aspect("equal")
ax.invert_xaxis()
ax.set_xlabel("RA [deg]"); ax.set_ylabel("Dec [deg]")
ax.set_title("(a) GOODS-S sky coverage per survey")
ax.legend(loc="lower left", fontsize=7)

# (b) z dist
ax = fig.add_subplot(gs[0, 1])
bins = np.arange(0.0, 6.0, 0.04)
for s in GS_SURVEYS:
    m = gs_specz["survey"] == s
    ax.hist(gs_specz["z"][m], bins=bins, histtype="step", lw=1.4, color=palette[s],
            label=f"{s} ({m.sum():,})")
shade_tier_windows(ax)
ax.set_yscale("log")
ax.set_xlim(0, 5.5)
ax.set_xlabel("Redshift"); ax.set_ylabel("N per \u0394z=0.04")
ax.set_title("(b) per-survey z distribution")
ax.legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "fig06_gs_unified")
plt.show()


In [ ]:
"""Figure 7 — Photo-z performance check: DJA spec-z vs ASTRODEEP photo-z.

ASTRODEEP photo-z systematically underestimates DJA spec-z at z > 1
(median \u0394z/(1+z) ~ +0.46). The implication for EMPOWER is that *tier-2 and
tier-3 candidates selected on photo-z alone* are unreliable until a true
spec-z is in hand for them. Uses the DJA-specific match radius.
"""
xm = blind_xmatch(dja_gs["ra"], dja_gs["dec"],
                  astrodeep["RA"], astrodeep["Dec"], radius_arcsec=match_radius("DJA"))
sec = (dja_gs["grade"] == 3) & (dja_gs["z_best"] > 0)
ok  = sec & xm.matched & (astrodeep["zphot"][xm.idx] > 0)
zd  = np.asarray(dja_gs["z_best"][ok])
zp  = np.asarray(astrodeep["zphot"][xm.idx[ok]])
dz  = (zd - zp) / (1.0 + zp)
tag = np.asarray(astrodeep["has_tag"])[xm.idx[ok]]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

ax = axes[0]
bins = np.linspace(-1.5, 1.5, 121)
ax.hist(np.clip(dz[tag],  bins[0], bins[-1]), bins=bins, histtype="step",
        lw=1.3, color="#1f77b4", label=f"prior spec-z tag ({tag.sum():,})")
ax.hist(np.clip(dz[~tag], bins[0], bins[-1]), bins=bins, histtype="step",
        lw=1.3, color="#d62728", label=f"photo-z only ({(~tag).sum():,})")
ax.set_xlabel("(z_DJA \u2212 z_phot) / (1 + z_phot)")
ax.set_ylabel("N pairs")
ax.set_title(f"(a) Photo-z residual against DJA spec-z  ({len(zd):,} pairs)")
ax.set_yscale("log")
ax.legend(fontsize=8)

ax = axes[1]
zmx = max(zd.max(), zp.max(), 4.0)
ax.scatter(zp[~tag], zd[~tag], s=4, c="#d62728", alpha=0.35, edgecolors="none",
           label="photo-z only")
ax.scatter(zp[tag],  zd[tag],  s=4, c="#1f77b4", alpha=0.35, edgecolors="none",
           label="prior spec-z tag")
ax.plot([0, zmx], [0, zmx], ls=":", c="k", lw=1)
ax.set_xlabel("z (ASTRODEEP photo-z)")
ax.set_ylabel("z (DJA spec-z, grade=3)")
ax.set_xlim(0, zmx); ax.set_ylim(0, zmx)
ax.set_aspect("equal", adjustable="box")
ax.set_title("(b) DJA vs ASTRODEEP zphot")
ax.legend(fontsize=8, loc="upper left")

fig.tight_layout()
save_fig(fig, "fig07_gs_photoz_check")
plt.show()
print(f"median \u0394z/(1+z) = {np.median(dz):+.3f}, "
      f"NMAD = {1.4826*np.median(np.abs(dz - np.median(dz))):.3f}")
print(xm.note)


In [ ]:
"""Per-tier verified-spec-z inventory in GOODS-S.

For each ASTRODEEP source we ask whether a member of the unified ``gs_specz``
sits within the matched survey's OWN radius (SPECZ_MATCH_RADII). We then count,
per EMPOWER tier:

* candidates   — ASTRODEEP zphot inside the tier and log M*>=10
* verified-z   — same, AND a unified spec-z within the per-survey radius
* still photoz — candidates with no spec-z match
"""
# Nearest gs_specz neighbour for every ASTRODEEP source; accept the match only if
# the separation is below the matched survey's own radius.
xm_gs = blind_xmatch(astrodeep["RA"], astrodeep["Dec"],
                     gs_specz["ra"], gs_specz["dec"], radius_arcsec=BASE_MATCH_RADIUS)
nn_survey   = np.asarray(gs_specz["survey"])[xm_gs.idx]
nn_radius   = np.array([match_radius(s) for s in nn_survey])
has_gsspecz = xm_gs.sep_arcsec < nn_radius
sp_survey   = np.where(has_gsspecz, nn_survey, "")

print("EMPOWER GOODS-S candidates (M*>=1e10, ASTRODEEP zphot inside tier):")
print(f"  {'tier':<14s}{'cands':>7s}{'verified':>9s}{'still photoz':>14s}"
      f"  surveys contributing")
for name in TIERS:
    m = tier_mask(astrodeep["zphot"], name) & (astrodeep["logM"] >= 10)
    n_cand = int(m.sum())
    n_ver  = int((m & has_gsspecz).sum())
    s_in   = sp_survey[m & has_gsspecz]
    bd = ", ".join(f"{s}:{(s_in==s).sum()}"
                   for s in GS_SURVEYS if (s_in==s).any())
    print(f"  {name:<14s}{n_cand:>7,}{n_ver:>9,}{n_cand-n_ver:>14,}  {bd}")

# gs_specz entries with no ASTRODEEP counterpart (no mass yet)
xm_inv     = blind_xmatch(gs_specz["ra"], gs_specz["dec"],
                          astrodeep["RA"], astrodeep["Dec"], radius_arcsec=BASE_MATCH_RADIUS)
inv_radius = np.array([match_radius(s) for s in np.asarray(gs_specz["survey"])])
gs_new     = ~(xm_inv.sep_arcsec < inv_radius)
print(f"\ngs_specz entries NOT in ASTRODEEP : {gs_new.sum():,}")
for s in GS_SURVEYS:
    print(f"   {s:<10s} {((gs_specz['survey'] == s) & gs_new).sum():>5,}")


## 4 · Synthesis

The figure below brings COSMOS and GOODS-S onto the same axes so the
field-level differences in spec-z completeness are easy to compare. Numbers
underneath each bar are the actual counts; bars are coloured by tier as used
throughout the notebook.

### Take-aways
* **COSMOS** is well covered for tier 1 ($\gtrsim 10^4$ Khostovan flag=4
  sources) and acceptably for tier 2; tier 3 is sparse and JADES-style
  programmes are essential. The KARMA H$\beta$+H$\alpha$ window holds
  $\sim 3{,}800$ Khostovan flag$\geq$3 sources.
* **GOODS-S** has *no* spec-z compilation of Khostovan quality; ASTRODEEP-GS43
  is the photometric reference only. The unified spec-z table built in §3.2
  delivers $\sim 6{,}900$ **unique** secure spec-z (JADES DR4 + DJA + VANDELS +
  MUSE-Wide + MUSE-UDF). It is heavily JWST-tilted at the expense of bright
  mass-selected galaxies, leaving $\sim 90\%$ of the $M_\star\!\geq\!10^{10}$
  EMPOWER candidates on photo-z only.
* **Per-source, not per-spectrum.** The compilation now collapses duplicates
  with a greedy unique-source pass using a *per-survey* match radius
  (`SPECZ_MATCH_RADII`: 0.5″ for JWST, 0.75″ for VIMOS/MUSE, 1.0″ for AAOmega).
  This is essential because DJA NIRSpec is per-spectrum — $\sim 8{,}800$ of its
  9,647 GOODS-S grade=3 rows are repeat gratings/visits of the same object, so
  an un-deduplicated stack roughly doubles the apparent catalogue size.
* **JADES DR4** (Curtis-Lake+25 / Scholtz+25), placed above DJA in priority,
  is now the single largest verified-z contributor in tier 2 and competitive in
  tiers 1/3 — but most of its robust redshifts were already absorbed by DJA
  v4.4, so its *net* gain over the prior union is only $\sim 140$ new sources.
* **Next priority for COSMOS**: cross-match the DJA grade=3 NEW set against
  COSMOS2020 for masses, then add into the EMPOWER selection pool. (Note the
  COSMOS DJA "NEW" count in §2.2 is still per-spectrum — collapse it the same
  way before use.)
* **Next priority for GOODS-S**: attach masses to the $\sim 2{,}300$ unified
  spec-z sources with no ASTRODEEP counterpart via a JADES NIRCam DR3/DR5
  photometric cross-match, and ingest the *value* redshifts for the surveys
  ASTRODEEP only tags (3D-HST grism, K20, Vanzella+08, VVDS, Kurk+12).

The blind RA/Dec cross-matches used here over-count chance matches; they are
adequate for these global counts but not for identifying individual targets.
A follow-up notebook will re-do the cross-matches against image-derived IDs
(COSMOS2020 / CANDELS-GS) before any source is added to the KMOS target list.


In [ ]:
"""Figure 8 — Combined COSMOS / GOODS-S target inventory per EMPOWER tier.

For each tier we plot the number of secure spec-z sources in (a) COSMOS
(Khostovan flag≥3) and (b) the unified GOODS-S compilation. The two fields are
on the same vertical scale so the size difference is immediately visible.
"""
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), sharey=True)

# (a) COSMOS
ax = axes[0]
counts_cos = []
for name in TIERS:
    counts_cos.append((is_secure & tier_mask(z, name)).sum())
counts_cos.append((is_secure & (z > KARMA_IZ_HB_HA[0])
                   & (z < KARMA_IZ_HB_HA[1])).sum())
names_x = list(TIERS.keys()) + ["KARMA IZ Hβ+Hα"]
colors_x = [TIERS[n]["color"] for n in TIERS] + ["navy"]
for x, c, lbl, n_v in zip(range(len(names_x)), colors_x, names_x, counts_cos):
    ax.bar(x, n_v, color=c, alpha=0.85, edgecolor="k", lw=0.4)
    ax.text(x, n_v, f"{n_v:,}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(names_x)))
ax.set_xticklabels(names_x)
ax.set_ylabel("N secure spec-z")
ax.set_title("(a) COSMOS — Khostovan flag ≥ 3")
ax.set_yscale("log")

# (b) GOODS-S unified
ax = axes[1]
counts_gs = []
for name in TIERS:
    counts_gs.append((tier_mask(gs_specz["z"], name)).sum())
counts_gs.append(((gs_specz["z"] > KARMA_IZ_HB_HA[0])
                  & (gs_specz["z"] < KARMA_IZ_HB_HA[1])).sum())
for x, c, lbl, n_v in zip(range(len(names_x)), colors_x, names_x, counts_gs):
    ax.bar(x, n_v, color=c, alpha=0.85, edgecolor="k", lw=0.4)
    ax.text(x, n_v, f"{n_v:,}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(names_x)))
ax.set_xticklabels(names_x)
ax.set_title("(b) GOODS-S — unified spec-z (DJA + VANDELS + MUSE)")

fig.suptitle("EMPOWER spec-z budget per tier — COSMOS vs GOODS-S", y=1.02)
fig.tight_layout()
save_fig(fig, "fig08_combined_summary")
plt.show()


### 4.1 What is *not* yet in this notebook
The following items are deliberately deferred — they should sit in dedicated
follow-up notebooks because they need more than a blind sky match:

1. **Image-matched source identification.** For both fields, sources picked up
   here as *new* should be confirmed against the parent photometric catalogue
   (COSMOS2020 for COSMOS, CANDELS-GS / JADES NIRCam for GOODS-S) using image
   centroids, not just RA/Dec.
2. **Spec-z values for ASTRODEEP-tagged sources.** ASTRODEEP-GS43 only tags
   that a prior survey observed a given source; the prior z value lives in the
   parent catalogue. Adding the 3D-HST grism, K20, VVDS and Vanzella+08
   redshifts directly would close most of the GOODS-S photo-z holes.
3. **Masses for the unified spec-z sources outside ASTRODEEP.** ~2,300 unique
   secure spec-z in GOODS-S have no ASTRODEEP counterpart — they need their own
   photometric crossmatch (JADES NIRCam DR3/DR5 or CANDELS-GS) before
   $M_\star\!\geq\!10^{10}$ can be applied.
4. **Per-source COSMOS DJA counts.** The COSMOS DJA gap analysis in §2.2 is
   still per-spectrum; apply the same `dedup_layers` collapse before using its
   "NEW" counts for selection.
5. **Manual line-ID review of catastrophic-z disagreements.** Pairs flagged in
   §2.2 and §3 with |Δz/(1+z)|>0.01 between any two surveys are candidates for
   manual review; they are not yet a science-grade conflict list.

### 4.2 Outputs
All figures referenced above are written to ``codes/figures/specz_analysis/``.
